In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/nitaisatapathy/welfake-dataset/WELFake_Dataset.csv


## Setting Up the Environment

In [2]:
!pip install torch torch-geometric spektral
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.4.0+${CUDA}.html

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 4.6 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.4.0+.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 7.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for torch-scatter: filename=torch_scatter-2.1.2-cp312-cp312-linux_x86_64.whl size=677287 sha256=55fcaf21c20e9be3520843ca5982f0b0ee8d1f223ed257c8e8376f682e72ce3b
  Stored in directory: /root/.cache/pip/wheels/84/20/50/44800723f57cd798630e77b3ec83bc80bd26a1e3dc3a672ef5
  Created wheel for torch-sparse: filename=torch_sparse-0.6.18-cp312-cp312-linux_x86_64.whl size=1262266 sha256=a0ca5b8ad0ae67bc93475d42f8b97e0eec12b44ac04f209175b6d9b1f0d982

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
import networkx as nx
from spektral.utils import normalized_adjacency

2026-04-01 06:10:04.035664: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775023804.269410      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775023804.337637      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775023804.927685      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775023804.927792      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775023804.927795      17 computation_placer.cc:177] computation placer alr

## Data Loading and Preprocessing

In [4]:
df = pd.read_csv('/kaggle/input/datasets/nitaisatapathy/welfake-dataset/WELFake_Dataset.csv')

df.shape

(72134, 4)

In [5]:
import string

def preprocess_text(text):
    # Convert the text to lowercase
    text = text.lower()
    # Remove all punctuation marks
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove extra whitespace (leading, trailing, and multiple spaces)
    text = ' '.join(text.split())
    return text

# Apply text preprocessing to the 'text' and 'title' columns
df['text'] = df['text'].astype(str).apply(preprocess_text)
df['title'] = df['title'].astype(str).apply(preprocess_text)

# Combine the 'title' and 'text' columns into a single 'content' column
df['content'] = df['title'] + " " + df['text']

# Sample a subset of the data to manage computational resources
sample_size = 20000
df_sample = df.sample(n=sample_size, random_state=42).reset_index(drop=True)

# Print the number of sampled articles
print(f"Sampled {sample_size} articles for analysis.")

Sampled 20000 articles for analysis.


In [6]:
df['label'].value_counts()

label
1    37106
0    35028
Name: count, dtype: int64

In [7]:
print("Sau khi lấy ngẫu nhiên 20000 mẫu để xây dựng Graph:")
df_sample.shape

Sau khi lấy ngẫu nhiên 20000 mẫu để xây dựng Graph:


(20000, 5)

In [8]:
df_sample['label'].value_counts()

label
1    10167
0     9833
Name: count, dtype: int64

In [9]:
# Extract the 'label' column as the target variable (y_data)
y_data = df_sample['label'].values  

# Drop the 'label' column from the DataFrame to avoid leakage during model training
df_sample.drop(['label'], axis=1, inplace=True)

# Display the extracted target variable
y_data

array([1, 1, 0, ..., 0, 0, 0])

In [10]:
df_sample.head(10)

,Unnamed: 0,title,text,content
0,61370,arnold schwarzenegger sends a message to liber...,,arnold schwarzenegger sends a message to liber...
1,2189,wow “we mexicans need to kill donald trump bef...,and now a message of peace and unity from one ...,wow “we mexicans need to kill donald trump bef...
2,60609,jimmy carter recovers from dehydration scare i...,winnipeg manitoba reuters former us president ...,jimmy carter recovers from dehydration scare i...
3,51565,2 friars’ mission reviving a brooklyn church i...,the two franciscan friars complete with robes ...,2 friars’ mission reviving a brooklyn church i...
4,39431,boy with autism makes his first friend ever an...,approximately 1 in 68 children has an autism s...,boy with autism makes his first friend ever an...
5,47839,something truly extraordinary and refreshing h...,the cnn debate between democratic candidates h...,something truly extraordinary and refreshing h...
6,42729,hashtag hell the fake left,by dady chery and gilbert mercierall writers w...,hashtag hell the fake left by dady chery and g...
7,37882,vice presidentelect pence says new hope dawns ...,washington reuters us vice presidentelect mike...,vice presidentelect pence says new hope dawns ...
8,10893,trump to republican senators dont leave town u...,washington reuters us president donald trump u...,trump to republican senators dont leave town u...
9,46977,white house obama may leave the country if tru...,0 comments the white house is refusing to deny...,white house obama may leave the country if tru...


## Feature Extraction

In [11]:
# Initialize the TF-IDF vectorizer with a limit of 2000 features
vectorizer = TfidfVectorizer(max_features=2000)  
# Transform the 'content' column into a TF-IDF feature matrix
tfidf_features = vectorizer.fit_transform(df_sample['content']).toarray()

from sklearn.preprocessing import StandardScaler

# Normalize the TF-IDF features using StandardScaler
scaler = StandardScaler()
tfidf_features = scaler.fit_transform(tfidf_features)

# Print the shape of the resulting TF-IDF feature matrix
print(f"TF-IDF feature matrix shape: {tfidf_features.shape}")

TF-IDF feature matrix shape: (20000, 2000)


In [12]:
tfidf_features

array([[-0.14634285, -0.25095506, -0.1805548 , ..., -0.11404131,
        -0.10414972, -0.11004868],
       [-0.14634285, -0.25095506, -0.1805548 , ..., -0.11404131,
        -0.10414972, -0.11004868],
       [-0.14634285, -0.25095506, -0.1805548 , ..., -0.11404131,
        -0.10414972, -0.11004868],
       ...,
       [-0.14634285, -0.25095506, -0.1805548 , ..., -0.11404131,
        -0.10414972, -0.11004868],
       [-0.14634285, -0.25095506, -0.1805548 , ..., -0.11404131,
        -0.10414972,  3.92091861],
       [-0.14634285, -0.25095506, -0.1805548 , ..., -0.11404131,
        -0.10414972, -0.11004868]])

## Constructing the Graph

In [13]:
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

k = 5  # Number of nearest neighbors

# Initialize NearestNeighbors with cosine similarity as the distance metric
nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='auto', metric='cosine').fit(tfidf_features)

# Find the k+1 nearest neighbors (including the point itself)
distances, indices = nbrs.kneighbors(tfidf_features)

# Initialize lists to store the rows and columns of the adjacency matrix
rows = []
cols = []

# Build the adjacency matrix by adding edges between the nodes and their neighbors
for i in range(tfidf_features.shape[0]):
    for j in range(1, k+1):  # Skip the first neighbor (itself)
        neighbor = indices[i][j]
        rows.append(i)
        cols.append(neighbor)

# Create a sparse adjacency matrix where each edge has a value of 1
data = np.ones(len(rows))
adjacency_sparse = csr_matrix((data, (rows, cols)), shape=(tfidf_features.shape[0], tfidf_features.shape[0]))

# Symmetrize the adjacency matrix to ensure the graph is undirected (bidirectional edges)
adjacency_sparse = adjacency_sparse.maximum(adjacency_sparse.transpose())

# Print the number of nodes and edges in the sparse adjacency matrix
print(f"Number of nodes: {adjacency_sparse.shape[0]}")
print(f"Number of edges: {adjacency_sparse.count_nonzero()}")

Number of nodes: 20000
Number of edges: 161485


In [14]:
# Normalize the adjacency matrix to improve convergence during training
A_norm_sparse = normalized_adjacency(adjacency_sparse, symmetric=True)

# Indicate that the adjacency matrix has been normalized
print("Adjacency matrix normalized.")

Adjacency matrix normalized.


## Prepare Tensors for Torch Modeling

In [15]:
import tensorflow as tf
import numpy as np
from spektral.utils import normalized_adjacency
from scipy.sparse import csr_matrix

# Use your actual normalized adjacency matrix (A_norm_sparse)
A_norm_sparse = csr_matrix(A_norm_sparse)  # Convert to CSR format if not already

# Convert the sparse matrix (csr_matrix) to a SparseTensor in TensorFlow
A_sparse_tensor = tf.sparse.from_dense(A_norm_sparse.toarray())  # Convert to dense, then to SparseTensor

# Normalize the adjacency matrix using Spektral (works with sparse matrices)
A_normalized = normalized_adjacency(A_norm_sparse)

# Convert the normalized sparse matrix to a dense numpy array
A_dense = A_normalized.toarray()  # Convert the csr_matrix to a dense array

# Convert the dense numpy array to a TensorFlow tensor
A_tensor = tf.convert_to_tensor(A_dense, dtype=tf.float32)

# Ensure X_tensor is a TensorFlow tensor and cast it to float32
X_tensor = tf.convert_to_tensor(tfidf_features, dtype=tf.float32)  # Use actual node features

# Convert labels to tensor (ensure the data type is float32 for consistency)
y_tensor = tf.convert_to_tensor(y_data, dtype=tf.float32)

# Optionally, check the shapes of the tensors for verification
print(A_tensor.shape)
print(X_tensor.shape)
print(y_tensor.shape)

2026-04-01 06:11:31.270971: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


(20000, 20000)
(20000, 2000)
(20000,)


## Splitting Data into Train and Test Sets

In [16]:
idx = np.arange(df_sample.shape[0])  # Create an array of indices for the dataset
# Split the indices into training and testing sets (70% train, 30% test)
idx_train, idx_test = train_test_split(idx, test_size=0.3, random_state=42)

# Create boolean masks for the training set (True for train, False for others)
train_mask = np.zeros(df_sample.shape[0], dtype=bool)
train_mask[idx_train] = True

# Create boolean masks for the testing set (True for test, False for others)
test_mask = np.zeros(df_sample.shape[0], dtype=bool)
test_mask[idx_test] = True

# Print the number of training and testing samples
print(f"Training samples: {train_mask.sum()}")
print(f"Testing samples: {test_mask.sum()}")

Training samples: 14000
Testing samples: 6000


## Train the Model

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
import torch_sparse

# Convert data to PyTorch tensors
X_tensor = torch.tensor(X_tensor.numpy(), dtype=torch.float32)  # Node features (X)
A_tensor = torch.tensor(A_tensor.numpy(), dtype=torch.float32)  # Adjacency matrix (A)
y_tensor = torch.tensor(y_tensor.numpy(), dtype=torch.long)    # Labels (y)

# PyTorch Geometric requires sparse adjacency matrix format, so convert it
from torch_sparse import coalesce
from torch_geometric.utils import dense_to_sparse

# Convert adjacency matrix to sparse COO format
edge_index, _ = dense_to_sparse(A_tensor)

# Create a PyTorch Geometric data object that contains the node features, edge indices, and labels
data = Data(x=X_tensor, edge_index=edge_index, y=y_tensor)

# Define the Graph Convolutional Network (GCN) model
class GCNModel(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(GCNModel, self).__init__()
        self.conv1 = GCNConv(in_channels, 128)  # First GCN layer with 128 output channels
        self.conv2 = GCNConv(128, out_channels)  # Second GCN layer with output channels (final prediction)
    
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)  # Apply first GCN layer
        x = torch.relu(x)  # Apply ReLU activation function
        x = self.conv2(x, edge_index)  # Apply second GCN layer
        return x

# Instantiate the GCN model, set the input channels and output classes (2 classes: real or fake)
model = GCNModel(in_channels=X_tensor.shape[1], out_channels=2)

# Define the optimizer and the loss function (CrossEntropyLoss for classification)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

# Training loop
epochs = 100
for epoch in range(epochs):
    model.train()  # Set the model to training mode
    optimizer.zero_grad()  # Zero the gradients for each epoch
    
    # Perform a forward pass
    out = model(data)
    
    # Calculate the loss using the true labels (data.y) and predicted outputs (out)
    loss = loss_fn(out, data.y)
    
    # Perform backward pass and optimization
    loss.backward()
    optimizer.step()
    
    # Print the loss every 10 epochs
    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

Epoch 0, Loss: 0.7395
Epoch 10, Loss: 0.3859
Epoch 20, Loss: 0.3338
Epoch 30, Loss: 0.3063
Epoch 40, Loss: 0.2872
Epoch 50, Loss: 0.2712
Epoch 60, Loss: 0.2566
Epoch 70, Loss: 0.2430
Epoch 80, Loss: 0.2300
Epoch 90, Loss: 0.2175


## Evaluate on Test Set

In [18]:
from sklearn.metrics import classification_report

# Make predictions
model.eval()
with torch.no_grad():
    out = model(data)
    _, predicted = torch.max(out, dim=1)  # Get the class with the highest score

# Generate the classification report
print(classification_report(data.y.cpu(), predicted.cpu(), target_names=["Class 0", "Class 1"]))

              precision    recall  f1-score   support

     Class 0       0.92      0.91      0.92      9833
     Class 1       0.92      0.92      0.92     10167

    accuracy                           0.92     20000
   macro avg       0.92      0.92      0.92     20000
weighted avg       0.92      0.92      0.92     20000

